#Environment Variables

In [1]:
import os

os.environ["HF_TOKEN"]="YOUR HF-TOKEN"
os.environ["COMET_API_KEY"]="YOUR-COMET-API-KEY"

In [2]:
HF_USERNAME="spiralMon"

#Imports

In [3]:
import comet_ml
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer,SFTConfig
from transformers import TrainingArguments,TextStreamer
from datasets import load_dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


#Login to Comet ML

In [4]:
comet_ml.login()

In [5]:
exp=comet_ml.start(project_name ="llm_twin_fine_tuned_instruct_model")

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/spiralmonster/llm-twin-fine-tuned-instruct-model/5ea21ad51c0b4c6d95a46c20ed31b576



#Load Model

In [6]:
max_seq_length=2048

In [7]:
model,tokenizer=FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    device_map="cuda"
)

==((====))==  Unsloth 2026.8.9: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


#Create PEFT Model

In [8]:
lora_rank=32
lora_alpha=32
lora_dropout=0
target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"]

In [9]:
model=FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=target_modules
)

Unsloth 2026.8.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


#Load Dataset

In [10]:
dataset_name="llm_twin_instruct_dataset"
dataset_id=HF_USERNAME+"/"+dataset_name

In [11]:
dataset=load_dataset(dataset_id)

In [12]:
dataset=dataset["train"]

#Format Dataset

In [13]:
alpaca_template="""
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}
"""

In [14]:
EOS_TOKEN=tokenizer.eos_token

In [15]:
def format_dataset(example):
  inputs=[]
  for instruction,output in zip(example["instructions"],example["outputs"]):
    inp=alpaca_template.format(instruction,output)+EOS_TOKEN
    inputs.append(inp)

  final_inputs={
      "text":inputs
  }
  return final_inputs

In [16]:
dataset=dataset.map(
    format_dataset,
    batched=True,
    remove_columns=dataset.column_names
)

In [17]:
dataset=dataset.train_test_split(test_size=0.05)

# Model Fine-Tuning

In [18]:
training_arguments=SFTConfig(
    output_dir="output",
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=True,
    dataset_num_proc=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    lr_scheduler_type="linear",
    num_train_epochs=4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    warmup_steps=10,
    report_to="comet_ml",
    seed=0
)

In [19]:
trainer=SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_arguments
)

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [20]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 149 | Num Epochs = 4 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)
COMET INFO: An experiment with the same configuration options is already running and will be reused.
COMET WARNING: String value length exceeds 1000 characters and will be truncated. Provided value: 'LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping={'base_model_class': 'LlamaForCausalLM', 'parent_library': 'transformers.models.llama.modeling_llama', 'unsloth_fixed': True}, peft_version='0.19.1', base_model_name_or_path='unsloth/Meta-Llama-3.1-8B-bnb-4bit', revision=None, inference_mode=False, r=32, target_modules={'gate_proj', 'v_proj', 'q_proj', 'o_proj', 'k_proj', 'down

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.612200
2,2.515200
3,2.614600
4,2.465400
5,2.339200
6,2.180400
7,2.027100
8,1.899700
9,1.830000
10,1.803200


TrainOutput(global_step=40, training_loss=1.7666860938072204, metrics={'train_runtime': 3871.2142, 'train_samples_per_second': 0.154, 'train_steps_per_second': 0.01, 'total_flos': 5.510733313012531e+16, 'train_loss': 1.7666860938072204, 'epoch': 4.0})

In [21]:
exp.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : disciplinary_frog_3896
COMET INFO:     url                   : https://www.comet.com/spiralmonster/llm-twin-fine-tuned-instruct-model/5ea21ad51c0b4c6d95a46c20ed31b576
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [33]                      : (0.1797829568386078, 0.7405576109886169)
COMET INFO:     train/epoch [41]               : (0.10666666666666667, 4.0)
COMET INFO:     train/grad_norm [40]           : (0.1783774197101593, 3.876332998275757)
COMET INFO:     train/learning_rate [40]       : (0.0, 0.0003)
COMET INFO:     train/loss [40]                : (1.4806, 2.6146)
COMET INFO:     train/total_flos               : 5.51073331301253

#Testing the Fine-Tuned Model

In [23]:
test_model=FastLanguageModel.for_inference(model)

In [24]:
model_input=alpaca_template.format(
    "Write an article about how gen-z are different",
    ""
)

In [25]:
model_input=tokenizer([model_input],return_tensors="pt").to("cuda")

In [26]:
text_streamer=TextStreamer(tokenizer)

In [27]:
output=test_model.generate(
    **model_input,
    streamer=text_streamer,
    max_new_tokens=256,
    use_cache=True
)

<|begin_of_text|>
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Write an article about how gen-z are different

### Response:

Gen Z, the generation born between 1995 and 2010, has been labeled as lazy, entitled, and unmotivated. However, this stereotype is far from the truth. Gen Z is the most educated and ambitious generation, and their work ethic is unmatched. They are not afraid to speak up and challenge authority, and they refuse to be silenced. They are the generation that has seen the most significant changes in the world, from the rise of social media to the global pandemic. They have witnessed the impact of technology and the changing nature of work, and they have adapted to these changes. Gen Z is not a problem, but a solution. They are the future, and we should embrace their unique perspective and skills.
<|end_of_text|>


#Publishing Model to Hugging Face Hub

In [28]:
model_name="Twin-LLM-Fine-Tuned-Instruct-Model-Llama-3.1-8B-bnb-4bit"
model_id=HF_USERNAME+"/"+model_name

In [29]:
model.save_pretrained_merged(
    "model",
    tokenizer,
    save_method="merged_16bit"

)

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 4 files from cache to `model`: 100%|██████████| 4/4 [06:30<00:00, 97.61s/it]


Successfully copied all 4 files from cache to `model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [06:59<00:00, 104.87s/it]


Unsloth: Merge process complete. Saved to `/content/model`


In [30]:
model.push_to_hub_merged(
    model_id,
    tokenizer,
    save_method="merged_16bit"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...B-bnb-4bit/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 4 files from cache to `spiralMon/Twin-LLM-Fine-Tuned-Instruct-Model-Llama-3.1-8B-bnb-4bit`: 100%|██████████| 4/4 [07:56<00:00, 119.21s/it]


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   3%|2         | 30.9MB / 1.17GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [17:24<00:00, 261.25s/it]


Unsloth: Merge process complete. Saved to `/content/spiralMon/Twin-LLM-Fine-Tuned-Instruct-Model-Llama-3.1-8B-bnb-4bit`
